# 07 · Uma ontologia que pessoas e agentes de IA conseguem ler

**Objetivo:** um agente de IA que vai consultar o grafo de uma planta tem o mesmo problema de um engenheiro no primeiro dia: não sabe quais classes existem, o que cada uma significa, nem como se ligam. Este notebook mostra como a ontologia responde a isso **sozinha**, e como garantimos que ela continue respondendo.

Quatro peças:

| Peça | Para quê | Onde |
|---|---|---|
| **Definição e exemplo em todo termo** | saber o que cabe em cada classe sem olhar os dados | `ontology/*.ttl` |
| **Catálogo** | consultar o modelo como texto, do tamanho de um prompt | `lab/catalog.py` |
| **Regras da casa (lint)** | impedir que o modelo apodreça em silêncio | `lab/lint.py` |
| **Perguntas de competência** | provar que o modelo responde ao que o negócio precisa | `lab/competency.py` |

Nenhum modelo de linguagem é chamado aqui. O notebook prepara o terreno: as funções abaixo são exatamente as **ferramentas** que um agente receberia.

### Antes de começar

| | |
|---|---|
| **Para quem** | Quem mantém um modelo de dados e quer que ele sirva de base para agentes de IA, ou só quer que ele não apodreça. |
| **Pré-requisitos** | Notebooks 01 a 06. Nenhum conhecimento de IA: nenhum modelo de linguagem é chamado. |
| **Tempo** | 35 minutos |

**Ao final você consegue:**

- Explicar por que definição e exemplo em cada termo mudam o que um agente consegue fazer.
- Usar o catálogo para descrever um termo e para gerar o modelo no tamanho de um prompt.
- Dizer o que o lint verifica e qual descuido ele pega que o RDF sozinho aceita.
- Explicar o que é uma pergunta de competência e por que ela vira teste.

**Onde isso aparece na fábrica:** Projetos de IA industrial travam menos por falta de modelo de linguagem e mais por falta de contexto: o agente não sabe o que os dados significam. Um modelo bem descrito é a parte que a fábrica precisa construir.

**Uma analogia para guardar:** É a diferença entre entregar a um estagiário as chaves do almoxarifado e entregar as chaves junto com o mapa, o catálogo das peças e a lista do que costuma ser pedido. O agente é esse estagiário, muito rápido e sem nenhuma memória da planta.

> **Como estudar:** rode uma célula por vez. Sempre que aparecer **Pare e preveja**, responda mentalmente antes de executar. Errar a previsão é o que faz o conceito fixar.

In [1]:
# Configuração: deixa a raiz do repositório no caminho de import
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option("display.max_colwidth", 80)
from lab.graph import load_ontology, load_plant, query, MFG, PRD, MNT, QA, NRG, INV, ID
from lab import catalog, lint, competency

## 1. O tamanho do modelo

In [2]:
m = pd.DataFrame(catalog.metrics())
print("total de termos:", m.terms.sum(), "| com definição:", m["with definition"].sum())
m

total de termos: 120 | com definição: 120


,module,classes,relations,attributes,with definition,terms
0,core,14,11,9,38,38
1,production,6,6,4,16,16
2,maintenance,8,7,5,20,20
3,quality,7,8,7,22,22
4,energy,4,4,3,11,11
5,inventory,5,6,2,13,13


Todo termo tem definição. Isso não é capricho: é a primeira regra do lint (seção 4), e é o que torna as próximas seções possíveis.

## 2. Perguntando ao modelo o que é uma coisa

`describe` devolve tudo sobre um termo: definição, exemplos, de quem herda, o que ele tem e quem aponta para ele, **incluindo o que vem das classes mãe**.

> **Pare e preveja.** `Downtime` foi declarado no arquivo de manutenção com apenas dois atributos próprios. Quantas coisas você acha que dá para saber sobre uma parada, contando o que ela herda?

In [3]:
print(catalog.describe("Downtime"))

mnt:Downtime  (class, module maintenance)
  An interval in which equipment did not run when it was scheduled to.
  Examples: filler-01 stopped 15 minutes for seal wear
  Is a: mfg:Event
  Has:
    mfg:endTime -> dateTime   When the event ended. Absent while the event is still open.
    mfg:occurredOn -> mfg:Asset   The asset the event happened to.
    mfg:performedBy -> mfg:Person   Who carried out the event.
    mfg:startTime -> dateTime   When the event began.
    mnt:causedBy -> mnt:FailureMode   The failure mode behind a stop.
    mnt:planned -> boolean   True for stops that were scheduled, such as lunch. Planned stops do not count against availability.
  Is pointed at by:
    mnt:resolves <- mnt:MaintenanceOrder   The stop a maintenance order was raised to fix.


**Leitura:** só com esse texto dá para escrever uma consulta sobre paradas sem abrir nenhum arquivo. `occurredOn`, `startTime` e `endTime` não foram declarados em `Downtime`: vieram de `mfg:Event`. Um agente que lesse só o arquivo de manutenção não saberia que uma parada tem horário.

Nome incompleto também é tratado, porque agentes (e pessoas) erram nomes:

In [4]:
print(catalog.describe("Faulted"))
print()
print(catalog.describe("Maintenance"))

mfg:Faulted  (state, module core)
  Halted by a fault. It must be reset, which takes it to STOPPED, before it can run again.
  Can transition to: mfg:Stopped

No term named 'Maintenance'. Did you mean: mnt:MaintenanceOrder, mnt:MaintenancePlan?


## 3. O modelo inteiro num cartão

`schema_card` gera o modelo como texto compacto. Cabe num prompt e pode ser recortado por módulo.

In [5]:
cartao = catalog.schema_card()
print(f"modelo completo: {len(cartao)} caracteres, cerca de {len(cartao) // 4} tokens\n")
print(catalog.schema_card(["maintenance"]))

modelo completo: 6722 caracteres, cerca de 1680 tokens

## maintenance
mnt:Component (is a mfg:Asset): A replaceable part of an equipment that maintenance tracks on its own.
mnt:CorrectiveOrder (is a mnt:MaintenanceOrder): Maintenance done because something failed.
mnt:Downtime (is a mfg:Event): An interval in which equipment did not run when it was scheduled to.
  .mnt:causedBy -> mnt:FailureMode
  .mnt:planned -> boolean
mnt:FailureMode: The way something fails. A shared vocabulary instead of free text, so stops can be counted and compared.
  .mnt:affects -> mnt:Component
mnt:MaintenanceOrder (is a mfg:Event): A piece of maintenance work carried out on an asset.
  .mnt:replaced -> mnt:Component
  .mnt:resolves -> mnt:Downtime
  .mnt:usedSpare -> mnt:SparePart
mnt:MaintenancePlan: A recurring preventive task for a component, with its interval and next due date.
  .mnt:covers -> mnt:Component
  .mnt:dueDate -> dateTime
  .mnt:intervalDays -> integer
  .mnt:lastExecuted -> dateTime
mnt:

Com esse cartão no contexto, um modelo de linguagem consegue escrever SPARQL usando os nomes certos de classes e relações, em vez de adivinhar. O cartão é **gerado** dos arquivos `.ttl`: se a ontologia muda, ele muda junto, e não existe documentação paralela para ficar desatualizada.

## 4. Regras da casa: o lint

O que o lint verifica: todo termo com rótulo e definição, toda classe com exemplo, toda propriedade com domínio e contradomínio, toda classe com pelo menos uma instância nos dados de exemplo, e **nenhum termo usado nos dados sem estar declarado**.

In [6]:
print("problemas no modelo atual:", lint.problems())

problemas no modelo atual:

 []


> **Pare e preveja.** Em RDF, nada impede alguém de gravar nos dados um predicado que não existe na ontologia. Você acha que isso gera erro em algum lugar?

In [7]:
# Simulando três descuidos comuns numa cópia do modelo
from rdflib import Graph, Literal, RDF, RDFS, OWL
from lab.graph import ROOT, PRD, MNT
modelo = load_ontology()
modelo.add((PRD.Pallet, RDF.type, OWL.Class)); modelo.add((PRD.Pallet, RDFS.label, Literal("Pallet")))   # classe sem definição, exemplo nem uso
modelo.add((MNT.severity, RDF.type, OWL.DatatypeProperty)); modelo.add((MNT.severity, RDFS.label, Literal("severity")))  # atributo sem nada
dados = Graph().parse(ROOT / "data" / "plant.ttl")
dados.add((ID["downtime/D-001"], MNT.rootCause, Literal("operador disse que foi a vedação")))            # predicado inventado na integração
for p in lint.problems(modelo, dados): print(" -", p)

 - prd:Pallet: no definition (rdfs:comment)
 - prd:Pallet: class without skos:example
 - mnt:severity: no definition (rdfs:comment)
 - mnt:severity: no range
 - mnt:severity: no domain
 - mnt:rootCause: used in the data but not declared in the ontology
 - prd:Pallet: no instance in the sample data (add one, or mark it abstract)


**Leitura:** o lint pegou a classe sem definição e sem uso, o atributo sem domínio nem contradomínio, e o caso mais traiçoeiro: `mnt:rootCause`, um predicado que **não existe na ontologia** mas apareceu nos dados. Em RDF nada impede isso; o dado entra, e ninguém mais sabe que ele existe. O lint roda no CI, então um pull request com qualquer um desses descuidos não passa.

## 5. Perguntas de competência

Uma pergunta de competência é uma pergunta de negócio escrita **antes** do modelo, usada para decidir o que entra nele e, depois, para provar que ele é suficiente. Aqui cada uma tem a consulta que a responde e uma verificação do resultado.

In [8]:
cq = pd.DataFrame(competency.run())
print(f"{cq.answered.sum()} de {len(cq)} perguntas respondidas")
cq[["id", "question", "modules", "answer"]]

15 de 15 perguntas respondidas


,id,question,modules,answer
0,CQ01,"What exists inside the São Paulo plant, at any depth?",core,"12 assets: 1 area, 2 lines, 6 machines and 3 components"
1,CQ02,"Which failure mode cost the most unplanned downtime, and on which line?","maintenance, core","Seal wear on Line 1: 3 stops, 54 min"
2,CQ03,"Which component does that failure mode hit, and was it replaced?",maintenance,"Seal Kit, replaced by MO-0077"
3,CQ04,"Which lots were made while those stops were happening, and what did inspecti...","maintenance, production, quality",L-1042 HOLD (1 stops); L-1043 REJECT (3 stops)
4,CQ05,"Which product lots consumed cap lot ML-2207, and how did they fare?","inventory, quality","L-1041 PASS, L-1042 HOLD, L-1043 REJECT"
5,CQ06,Which line uses the most energy per thousand good parts?,"energy, production","Line 3: 5.76 kWh per 1,000"
6,CQ07,Where do I subscribe to read the live state of every filler?,core,"3 topics, for example acme/curitiba/packaging/line-3/filler-01/state"
7,CQ08,On which lines can orange juice 1 l run?,"core, production",only Line 3
8,CQ09,Was any preventive plan overdue on the day of the shift?,maintenance,"PM-SEAL-L1 for the Seal Kit of Filler L1, due 2026-10-28; that component cau..."
9,CQ10,"Which lots have cap torque below specification, and which cap lot did they use?","quality, inventory","L-1042: 2 of 5 samples, cap lot ML-2207; L-1043: 4 of 5 samples, cap lot ML-..."


**Por que isso importa:** se alguém renomear uma relação ou remover uma classe, a pergunta correspondente deixa de ser respondida e o teste diz **qual pergunta de negócio foi perdida**, não só "um teste falhou". A coluna `modules` mostra que quase todas cruzam mais de um domínio, que é a razão de existir do grafo.

## 6. A ontologia dentro do Neo4j

Na exportação para Neo4j (notebook 05) iam só os dados. Agora o próprio modelo vai junto, como nós: cada classe vira um nó com definição e exemplos, e cada instância aponta para a sua classe.

In [9]:
from lab.to_cypher import ontology_to_cypher
linhas = ontology_to_cypher()
print(len(linhas), "comandos\n")
amostra = [linhas[1]] + [x for x in linhas if "SUBCLASS_OF" in x][:1] + [x for x in linhas if "[:DOMAIN]" in x][:1] + [x for x in linhas if "CAN_TRANSITION_TO" in x][:1] + [linhas[-1]]
for l in amostra:
    print(l[:260]); print()

260 comandos

MERGE (n:OntologyTerm:OntologyClass {iri: 'https://w3id.org/industrial-ontology-lab/core#Area'}) SET n.name = 'Area', n.module = 'core', n.definition = 'A part of a site organized around one kind of work.', n.examples = ['packaging, utilities, warehouse'];

MATCH (a:OntologyTerm {iri: 'https://w3id.org/industrial-ontology-lab/core#Area'}), (b:OntologyTerm {iri: 'https://w3id.org/industrial-ontology-lab/core#Asset'}) MERGE (a)-[:SUBCLASS_OF]->(b);

MATCH (a:OntologyTerm {iri: 'https://w3id.org/industrial-ontology-lab/core#canTransitionTo'}), (b:OntologyTerm {iri: 'https://w3id.org/industrial-ontology-lab/core#OperatingState'}) MERGE (a)-[:DOMAIN]->(b);

MATCH (a:OntologyTerm {iri: 'https://w3id.org/industrial-ontology-lab/core#Faulted'}), (b:OntologyTerm {iri: 'https://w3id.org/industrial-ontology-lab/core#Stopped'}) MERGE (a)-[:CAN_TRANSITION_TO]->(b);

MATCH (n:Resource) UNWIND labels(n) AS l MATCH (c:OntologyClass {name: l}) MERGE (n)-[:INSTANCE_OF]->(c);



Com o modelo no banco, quem consulta o Neo4j pode perguntar primeiro **o que existe e o que significa**, e só depois buscar as instâncias:

```cypher
// o que é um Downtime e o que posso seguir a partir dele?
MATCH (c:OntologyClass {name: 'Downtime'})-[:SUBCLASS_OF*0..]->(k)<-[:DOMAIN]-(p:OntologyProperty)
RETURN c.definition, p.name, p.definition;

// todas as instâncias de qualquer tipo de evento
MATCH (n)-[:INSTANCE_OF]->(:OntologyClass)-[:SUBCLASS_OF*0..]->(:OntologyClass {name: 'Event'}) RETURN n LIMIT 25;
```

## 7. Como um agente usaria isto

O roteiro abaixo imita o ciclo de um agente investigando "por que o lote L-1043 foi reprovado?". **Aqui quem decide o próximo passo é o roteiro**; num agente de verdade seria um modelo de linguagem. O que interessa é ver que cada passo só usa as ferramentas deste notebook.

In [10]:
from IPython.display import display
from lab import queries as q
g = load_plant()

def ferramenta_descrever(termo): return catalog.describe(termo)
def ferramenta_consultar(sparql): return pd.DataFrame(query(g, sparql))

passos = [
    ("O que é um lote e o que se liga a ele?", lambda: print(ferramenta_descrever("Lot"))),
    ("Quem inspecionou o lote e o que decidiu?", lambda: display(ferramenta_consultar(q.LOTS_HIT_BY_FAILURE).query("lot == 'L-1043'"))),
    ("A inspeção se liga a medições. O que é uma medição?", lambda: print(ferramenta_descrever("Measurement"))),
    ("Como ficou o torque desse lote contra a especificação?", lambda: display(ferramenta_consultar(q.TORQUE_VS_SPEC).query("lot == 'L-1043'"))),
    ("O lote veio de uma corrida que consumiu material. Onde está o resto?", lambda: display(ferramenta_consultar(q.STOCK_TO_QUARANTINE))),
]
for i, (pensamento, acao) in enumerate(passos, 1):
    print(f"\n--- passo {i}: {pensamento}")
    acao()


--- passo 1: O que é um lote e o que se liga a ele?
prd:Lot  (class, module production)
  The traceable quantity of product made in one run. Quality decisions and recalls are taken per lot.
  Examples: L-1042
  Has:
    prd:lotOf -> prd:Product   The product a lot contains.
  Is pointed at by:
    prd:produced <- prd:ProductionRun   The lot that came out of a run.
    qa:concerns <- qa:NonConformance   The lot a non-conformance is about.
    qa:inspects <- qa:Inspection   The lot an inspection examined.

--- passo 2: Quem inspecionou o lote e o que decidiu?


,lot,run,result,leaks,stops_during_run,failure_modes
2,L-1043,R-1043,REJECT,61.0,3,"Seal wear, Bottle jam"



--- passo 3: A inspeção se liga a medições. O que é uma medição?
qa:Measurement  (class, module quality)
  One measured value of a characteristic, taken during an inspection.
  Examples: 1.31 N·m on a sample of lot L-1043
  Has:
    qa:value -> decimal   The measured value, in the unit of the characteristic.
  Is pointed at by:
    qa:hasMeasurement <- qa:Inspection   The values measured during an inspection.

--- passo 4: Como ficou o torque desse lote contra a especificação?


,lot,line,material_lot,lower,upper,samples,mean,lowest,out_of_spec
2,L-1043,Line 1,ML-2207,1.40,2.00,5,1.36,1.31,4



--- passo 5: O lote veio de uma corrida que consumiu material. Onde está o resto?


,material_lot,supplier,location,where,quantity,because_of
0,ML-2207,Cap Co,Warehouse rack A-03,São Paulo plant,12000,"NC-001, NC-002"
1,ML-2207,Cap Co,"Point of use, line 1",Line 1,1800,"NC-001, NC-002"


Cada passo fez uma de duas coisas: **ler o modelo** para saber o que existe, ou **consultar os dados** usando o que acabou de ler. É esse vaivém que um agente faz, e ele só funciona se o modelo tiver definições, se o catálogo existir e se as consultas tiverem resposta conhecida para comparar.

## Resumo

* **Definição e exemplo em todo termo** transformam a ontologia em documentação que não desatualiza.
* O **catálogo** entrega o modelo como texto, por termo ou inteiro, no tamanho de um prompt.
* O **lint** impede os três descuidos clássicos, inclusive o termo usado nos dados sem ter sido declarado.
* As **perguntas de competência** dizem o que o modelo precisa responder, e viram teste.
* Tudo isso é a base de um agente que investiga com trilha auditável, que é o próximo projeto.

---
## Erros comuns

- **Documentar o modelo num arquivo à parte.** Ele desatualiza na primeira mudança. A definição mora no próprio termo e a documentação é gerada.
- **Entregar o grafo ao agente sem o esquema.** Ele inventa nomes de classes e relações. O cartão do modelo no contexto resolve a maior parte disso.
- **Definição que repete o nome.** "Downtime: a downtime" não ajuda ninguém. A definição diz o que entra e o que não entra, e o exemplo confirma.
- **Testar só se a consulta roda.** Ela pode rodar e responder errado. A pergunta de competência verifica a resposta.

## Teste seu entendimento

Responda antes de abrir cada resposta.

**1. Por que o cartão do modelo é gerado dos arquivos `.ttl` em vez de escrito à mão?**

<details><summary>Ver resposta</summary>

Para nunca divergir da ontologia. Se um termo muda, o cartão muda junto na próxima geração.

</details>

**2. Qual descuido o lint pega que nenhum validador de RDF acusaria?**

<details><summary>Ver resposta</summary>

Um termo usado nos dados sem estar declarado na ontologia. O dado é RDF válido, entra no grafo, e ninguém fica sabendo que existe.

</details>

**3. O que uma pergunta de competência entrega que um teste comum não entrega?**

<details><summary>Ver resposta</summary>

Quando falha, ela diz qual pergunta de negócio deixou de ser respondida. Isso dá o impacto da mudança em termos que o negócio entende.

</details>

## Exercícios para praticar

1. Use `catalog.describe` em três termos que você não conhece e escreva uma consulta só com o que leu.
2. Acrescente uma classe ao modelo sem exemplo e rode `python -m lab.lint`. Depois corrija.
3. Escreva uma pergunta de competência nova, a consulta e a verificação, e inclua em `lab/competency.py`.
4. Gere o cartão só de `core` e `quality` e estime quantos tokens ele ocupa.

## Glossário

| Termo | Significado |
|---|---|
| **Pergunta de competência** | Pergunta de negócio que o modelo precisa conseguir responder; guia o projeto e vira teste. |
| **Catálogo** | Visão do modelo como lista de termos com definição, exemplos e ligações. |
| **Cartão do esquema** | O modelo como texto compacto, para caber no contexto de um modelo de linguagem. |
| **Lint** | Verificação automática de regras de estilo e consistência. |
| **Metamodelo** | O modelo descrito como dado: classes e relações como nós do próprio grafo. |
| **Ferramenta (de agente)** | Função que um agente de IA pode chamar para ler ou consultar algo. |

**Próximo passo:** Você fechou a trilha de ontologia. O [ot-security-lab](https://github.com/MatheusEduGarcia/ot-security-lab) mostra como proteger a rede por onde esses dados passam.